In [2]:
from catanatron import Game, RandomPlayer, Color

### Useful code for starting a game

In [3]:
# Play a simple 4v4 game
players = [
    RandomPlayer(Color.RED),
    RandomPlayer(Color.BLUE),
    RandomPlayer(Color.WHITE),
    RandomPlayer(Color.ORANGE),
]

In [191]:
game = Game(players)
# print(game.play())  # returns winning color

#### Map Coordinates

X axis = East/West,
Y axis = Northwest/Southeast,
Z axis = Northeast/Southwest,

            (0,2,-2)   (1,1,-2)   (2,0,-2)
         (-1,2,-1)   (0,1,-1)   (1,0,-1)   (2,-1,-1)
    (-2,2,0)   (-1,1,0)   (0,0,0)    (1,-1,0)   (2,-2,0)
         (-2,1,1)   (-1,0,1)   (0,-1,1)   (1,-2,1)
              (-2,0,2)   (-1,-1,2)  (0,-2,2)

In [5]:
## Implementation of our map
from catanatron.models.map import CatanMap, initialize_tiles, BASE_MAP_TEMPLATE, TOURNAMENT_MAP, build_map, SHEEP, ORE, WHEAT, WOOD, BRICK

## My own map based on a tournament

## US CNC 2024 Prelim #3 Map
# https://www.reddit.com/r/Catan/comments/1fbi61s/catan_us_national_championship_maps/#lightbox

US_CNC_PRELIM_3_TILES = initialize_tiles(
    BASE_MAP_TEMPLATE,
    
    # NUMBERS (same pattern as tiles)
    [11, 8, 3, 6, 10, 5, 9, 4, 10, 3, 9, 6,    5, 4, 12, 11, 8, 2],
    # PORTS (none = 3:1) (clockwise from far right)
    [
        ORE,
        WHEAT,
        None,
        WOOD,
        BRICK,
        None,
        None,
        SHEEP,
        None, # far right of board
    ],
    # RESOURCE TILES (clockwise spiraling out)
    [
        WHEAT,
        BRICK,
        SHEEP,
        WOOD,
        SHEEP,
        ORE,
        BRICK,
        SHEEP,
        WHEAT,
        WOOD,
        WHEAT,
        ORE,
        # layer 1
        WOOD,
        WHEAT,
        WOOD,
        ORE,
        SHEEP,
        BRICK,
        # layer 0
        None,
    ],
)

US_CNC_PRELIM_3_MAP = CatanMap.from_tiles(US_CNC_PRELIM_3_TILES)
BASE_MAP = build_map("BASE")

In [213]:
game = Game(players, seed = 42, catan_map = TOURNAMENT_MAP)
# game = Game(players, seed = 42, catan_map = BASE_MAP)
# game = Game(players, seed = 42, catan_map = US_CNC_PRELIM_3_MAP)

In [9]:
# allows us to see the game as it's being played and the result
from catanatron.web.utils import open_link
open_link(game)  # opens game in browser

In [214]:
print(game.state)
action = game.play_tick()
print(action)

ActionRecord(action=Action(color=C.RED, action_type=AT.BUILD_SETTLEMENT, value=37), result=None)


In [215]:
while game.winning_color() is None:
    print(game.state)
    action = game.play_tick()
    print(action)

ActionRecord(action=Action(color=C.RED, action_type=AT.BUILD_ROAD, value=(14, 37)), result=None)
ActionRecord(action=Action(color=C.ORANGE, action_type=AT.BUILD_SETTLEMENT, value=0), result=None)
ActionRecord(action=Action(color=C.ORANGE, action_type=AT.BUILD_ROAD, value=(0, 1)), result=None)
ActionRecord(action=Action(color=C.BLUE, action_type=AT.BUILD_SETTLEMENT, value=52), result=None)
ActionRecord(action=Action(color=C.BLUE, action_type=AT.BUILD_ROAD, value=(23, 52)), result=None)
ActionRecord(action=Action(color=C.WHITE, action_type=AT.BUILD_SETTLEMENT, value=27), result=None)
ActionRecord(action=Action(color=C.WHITE, action_type=AT.BUILD_ROAD, value=(8, 27)), result=None)
ActionRecord(action=Action(color=C.WHITE, action_type=AT.BUILD_SETTLEMENT, value=13), result=None)
ActionRecord(action=Action(color=C.WHITE, action_type=AT.BUILD_ROAD, value=(13, 14)), result=None)
ActionRecord(action=Action(color=C.BLUE, action_type=AT.BUILD_SETTLEMENT, value=33), result=None)
ActionRecord(acti

### Building the Observation Space

##### ASSUME SELF is BLUE and Opponent is RED

In [10]:
import numpy as np
from catanatron.models.enums import ActionType

#### 1: Hexes (6 nodes x 19 hexes)

In [212]:
from catanatron.models.map import DICE_PROBAS
from catanatron.models.enums import ActionType, RESOURCES, ActionPrompt, SETTLEMENT, CITY, DEVELOPMENT_CARDS, VICTORY_POINT

In [ ]:
def get_hex_features(game) -> list:
    hex_features = []
    robber_coord = game.state.board.robber_coordinate
    for coord, tile in game.state.board.map.land_tiles.items(): #19 hexes in catan
        tile_prob = DICE_PROBAS[tile.number]
        tile_resource = tile.resource

        # resource pip data
        resource_pips = [tile_prob  if resource==tile_resource else 0.0 for resource in RESOURCES]

        # robber data
        has_robber = 0
        if coord == robber_coord:
            has_robber = 1
        
        hex_features.extend(resource_pips + [has_robber])
    
    return hex_features

len(get_hex_features(game))

114

#### 2: Vertices (16 nodes x 19 vertices)

In [13]:
c = game.state.board.map.node_production

In [14]:
c

{0: Counter({'WHEAT': 0.1111111111111111, 'ORE': 0.08333333333333333}),
 1: Counter({'ORE': 0.1388888888888889, 'WHEAT': 0.1111111111111111}),
 2: Counter({'ORE': 0.1388888888888889, 'WOOD': 0.05555555555555555}),
 3: Counter({'WOOD': 0.05555555555555555, 'ORE': 0.05555555555555555}),
 4: Counter({'BRICK': 0.1111111111111111, 'ORE': 0.05555555555555555}),
 5: Counter({'BRICK': 0.1111111111111111, 'ORE': 0.08333333333333333}),
 6: Counter({'ORE': 0.1388888888888889,
          'WHEAT': 0.1111111111111111,
          'WOOD': 0.08333333333333333}),
 7: Counter({'ORE': 0.1388888888888889,
          'WHEAT': 0.1111111111111111,
          'WOOD': 0.08333333333333333}),
 8: Counter({'ORE': 0.1388888888888889,
          'WHEAT': 0.1111111111111111,
          'SHEEP': 0.027777777777777776}),
 9: Counter({'ORE': 0.1388888888888889,
          'WOOD': 0.05555555555555555,
          'SHEEP': 0.027777777777777776}),
 10: Counter({'WOOD': 0.05555555555555555,
          'BRICK': 0.05555555555555555,
   

In [17]:
game.state.board.buildings

{37: (C.RED, 'CITY'),
 0: (C.ORANGE, 'CITY'),
 52: (C.BLUE, 'SETTLEMENT'),
 27: (C.WHITE, 'SETTLEMENT'),
 13: (C.WHITE, 'SETTLEMENT'),
 33: (C.BLUE, 'SETTLEMENT'),
 9: (C.ORANGE, 'CITY'),
 7: (C.RED, 'CITY'),
 22: (C.RED, 'CITY'),
 35: (C.RED, 'SETTLEMENT'),
 50: (C.RED, 'SETTLEMENT')}

In [18]:
SELF_COLOR = Color.BLUE
OPP_COLOR = Color.RED
TILE_ID_TO_COORDINATE = {tile.id: coordinate for coordinate, tile in game.state.board.map.land_tiles.items()} 

In [ ]:
def get_vertex_features(game):
    vertex_features = []
    
    game_buildings: dict[int, tuple[Color, str]] = game.state.board.buildings
    resource_to_port_node: dict[str, set[int]]=  game.state.board.map.port_nodes
    port_node_to_resource: dict[int, str] = {node_id : resource for resource, id_set in resource_to_port_node.items() for node_id in id_set}
    
    is_init_settlement_period: bool = game.state.current_prompt == ActionPrompt.BUILD_INITIAL_SETTLEMENT
    robber_coord = game.state.board.robber_coordinate

    for id, pip_counts in game.state.board.map.node_production.items(): # node production unaffected by the robber

        # settlement status
        if id in game_buildings:
            building_color, building_type = game_buildings[id]
            settlement_status = 0.5 if building_type == SETTLEMENT else 1.0
            if building_color == OPP_COLOR: 
                settlement_status *= -1
        else:
            settlement_status = 0.0


        # resources per turn
        resource_pips = [pip_counts[resource] if resource in pip_counts else 0.0 for resource in RESOURCES]
        total_pips = sum(pip_counts.values())

        # port resource improvement
        if id not in port_node_to_resource:
            port_trade = [0] * 5
        else:
            port_type = port_node_to_resource[id]
            if port_type is None:
                port_trade = [1] * 5
            else:
                port_trade = [2.0 if port_type == resource else 0.0 for resource in RESOURCES]
        
        # impacted by robber?
        impacted_by_robber = 0
        for tile in game.state.board.map.adjacent_tiles[id]:
            if tile.resource is not None and TILE_ID_TO_COORDINATE[tile.id] == robber_coord:
                impacted_by_robber = 1
        
        # can be built on in general?
        is_buildable = 1.0 if id in game.state.board.board_buildable_ids else 0.0

        # TODO -> THIS REQUIRES A LARGE SEARCH FOR EACH VERTEX, WHICH WOULD SLOW EVERYTHING DOWN... WILL SKIP FOR NOW
        # distance from self network (road # / 15 for normalization to below 1)
        
        # distance from opp network (road # / 15 for normalization to below 1)

        vertex_features.extend([settlement_status] + resource_pips + [total_pips] + port_trade + [impacted_by_robber, is_buildable])
    
    return vertex_features

In [21]:
len(get_vertex_features(game)) # should be 864 but left out distance to settlement nodes for now

756

#### 3: Edges

In [22]:
from catanatron.features import get_edges, get_node_distances
from catanatron.models.board import longest_acyclic_path # use for longest road

In [23]:
def get_edge_features(game):
    
    edge_features = []

    self_buildable_edges = game.state.board.buildable_edges(SELF_COLOR)
    opp_buildable_edges = game.state.board.buildable_edges(OPP_COLOR)
    for edge in get_edges():

        # settlement status
        road_color = game.state.board.get_edge_color(edge)
        if road_color is None:
            settlement_status = 0
        elif road_color == SELF_COLOR:
            settlement_status = 1
        else:
            settlement_status = -1
    
        self_can_build_now = edge in self_buildable_edges
        opp_can_build_now = edge in opp_buildable_edges

        # TODO requires significant computation (is it worth it?)
        # self_extends_longest_road = 
        # opp_extends_longest_road = 

        # TODO -> is this that useful?
        is_adjacent_vertex_available = 1.0 if (edge[0] in game.state.board.board_buildable_ids or edge[1] in game.state.board.board_buildable_ids) else 0.0

        edge_features.extend([settlement_status, self_can_build_now, opp_can_build_now, is_adjacent_vertex_available])
    
    return edge_features
        
# game.state.board.road_lengths -> Useful dictionary of each player's longest road length
# game.state.board.continuous_roads_by_player(Color.WHITE) TODO -> modify this function to give all possible longest length paths and not just one per connected component
# game.state.board.buildable_edges(Color.ORANGE) -> TODO -> use this with the above functions to make the extends longest road 
len(get_edge_features(game))

288

#### 4: Hand Features

In [245]:
from catanatron.state import PLAYER_INITIAL_STATE

In [ ]:
def get_hand_features(game):
    
    player_states = game.state.player_state

    color_to_index = game.state.color_to_index # map player color to player index in the game
    self_index = color_to_index[SELF_COLOR]
    opp_index = color_to_index[OPP_COLOR]

    # Self Hand

    # amt of resource in self hand
    self_resource_counts = [player_states[f"P{self_index}_{resource}_IN_HAND"] for resource in RESOURCES]
    self_total_resources = sum(self_resource_counts)
    self_over_card_limit = int(self_total_resources > game.state.discard_limit)

    # development cards
    self_dev_card_counts = [player_states[f"P{self_index}_{resource}_IN_HAND"] for resource in DEVELOPMENT_CARDS]
    self_total_dev_cards = sum(self_dev_card_counts)

    # buildings still available
    self_remaining_buildings = [player_states[f"P{self_index}_{building}_AVAILABLE"] for building in ["ROADS", "SETTLEMENTS", "CITIES"]]

    # Opponent hand
    opp_resource_counts = [player_states[f"P{opp_index}_{resource}_IN_HAND"] for resource in RESOURCES]
    opp_total_resources = sum(opp_resource_counts)
    opp_over_card_limit = int(opp_total_resources > game.state.discard_limit)

    # dev cards (only use total for opponent)
    opp_dev_card_counts = [player_states[f"P{opp_index}_{resource}_IN_HAND"] for resource in DEVELOPMENT_CARDS]
    opp_total_dev_cards = sum(opp_dev_card_counts)

    opp_remaining_buildings = [player_states[f"P{opp_index}_{building}_AVAILABLE"] for building in ["ROADS", "SETTLEMENTS", "CITIES"]]

    return [
        # Self hand
        *self_resource_counts,
        self_total_resources, self_over_card_limit,
        *self_dev_card_counts,
        self_total_dev_cards,
        *self_remaining_buildings,

        # Opp hand
        *opp_resource_counts,
        opp_total_resources, opp_over_card_limit,
        opp_total_dev_cards,
        *opp_remaining_buildings,
    ]

len(get_hand_features(game))


27

#### 5: Strategic Features

In [148]:
from catanatron.features import build_production_features, get_node_production

In [179]:
from catanatron.state_functions import get_player_buildings

def get_player_production(game, player_color, consider_robber=False):
    board = game.state.board
    productions = []

    for resource in RESOURCES:
        production = 0
        for node_id in get_player_buildings(game.state, player_color, SETTLEMENT):
            adj_tiles = board.map.adjacent_tiles[node_id]
            production += sum([DICE_PROBAS[t.number] for t in adj_tiles 
                               if t.resource == resource and 
                                    (not consider_robber or
                                    (consider_robber and TILE_ID_TO_COORDINATE[t.id] != board.robber_coordinate))
                            ])
            
        for node_id in get_player_buildings(game.state, player_color, CITY):
            adj_tiles = board.map.adjacent_tiles[node_id]
            production += 2*sum([DICE_PROBAS[t.number] for t in adj_tiles 
                               if t.resource == resource and 
                                    (not consider_robber or
                                    (consider_robber and TILE_ID_TO_COORDINATE[t.id] != board.robber_coordinate))
                            ])

        productions.append(production)
    
    return productions

get_player_production(game, SELF_COLOR, consider_robber=True)

[0, 0, 0.2222222222222222, 0, 0]

In [67]:
def compute_income_diversity(pip_production: list):
    pips = np.array(pip_production, dtype=float)
    total = pips.sum()

    if total == 0: 
        return 0.0

    probabilities = pips/total
    probabilities = probabilities[probabilities>0]

    entropy = -np.sum(probabilities * np.log(probabilities))

    max_entropy = np.log(len(pips))
    norm_entropy = entropy/max_entropy

    return norm_entropy


In [ ]:
def get_player_numbers(game, player_color):
    """
    Returns a dict of {number: total_pips} showing which
    dice rolls produce resources for the player.
    """
    board = game.state.board
    buildings = game.state.buildings_by_color[player_color]
    
    number_coverage = {n: 0 for n in range(2, 13) if n != 7}
    
    for building_type, node_ids in buildings.items():

        if building_type == "ROAD":
            continue

        # Multiplier: settlement = 1, city = 2
        multiplier = 2 if building_type == CITY else 1

        for node_id in node_ids:
            # Get adjacent tiles for this building
            tiles = board.map.adjacent_tiles[node_id]
        
            for tile in tiles:
                if tile.number is not None:  # Ignore desert
                    number_coverage[tile.number] += multiplier
    
    return number_coverage

def compute_roll_diversity(game, player_color):
    player_roll_counts = get_player_numbers(game, player_color=player_color)

    values = np.array(list(player_roll_counts.values()), dtype=float)
    covered = values[values > 0]
    total_values = sum(covered)
    probs = covered/total_values
    number_entropy = -np.sum(probs * np.log(probs))
    number_diversity = number_entropy / np.log(len(values))

    return number_diversity

In [ ]:

CLAIM_ARMY_SIZE = 3
def get_strategic_features(game):
    
    player_states = game.state.player_state
    
    # NOTE -> FOUND A BUG IN THE CATANOTRON IMPLEMENTATION OF THIS
    # production_features = build_production_features(consider_robber=False)
    # player_production = production_features(game, SELF_COLOR)
    # rob_impact_production_features = build_production_features(consider_robber=True)
    # rob_impact_player_production = production_features(game, SELF_COLOR)

    color_to_index = game.state.color_to_index # map player color to player index in the game
    self_index = color_to_index[SELF_COLOR]
    opp_index = color_to_index[OPP_COLOR]

    # Victory Points
    self_VP = player_states[f"P{self_index}_VICTORY_POINTS"]
    opp_VP = player_states[f"P{opp_index}_VICTORY_POINTS"]
    self_VP_to_win = game.vps_to_win - self_VP
    opp_VP_to_win = game.vps_to_win - opp_VP
    VP_diff = self_VP - opp_VP

    # Roads
    self_longest_road_length = game.state.board.road_lengths[SELF_COLOR]
    opp_longest_road_length = game.state.board.road_lengths[OPP_COLOR]
    who_has_longest_road: int = player_states[f"P{self_index}_HAS_ROAD"] - player_states[f"P{opp_index}_HAS_ROAD"]
    # TODO -> should it be longest + 1 if someone else already has it? Does it matter?
    self_dist_to_longest_road = game.state.board.road_length - self_longest_road_length
    opp_dist_to_longest_road = game.state.board.road_length - opp_longest_road_length
    road_length_diff = self_longest_road_length - opp_longest_road_length

    # Army
    self_army_size = player_states[f"P{self_index}_PLAYED_KNIGHT"]
    opp_army_size = player_states[f"P{opp_index}_PLAYED_KNIGHT"]
    who_has_largest_army: int = player_states[f"P{self_index}_HAS_ARMY"] - player_states[f"P{opp_index}_HAS_ARMY"]
    army_diff = self_army_size - opp_army_size
    # TODO -> should it be longest + 1 if someone else already has it? Does it matter?
    self_dist_to_largest_army = CLAIM_ARMY_SIZE-self_army_size if who_has_largest_army == 0 else 0 if who_has_largest_army == 1 else -(army_diff-1) # -1 accounts for needing to beat the record rather than just match to claim it
    opp_dist_to_largest_army = CLAIM_ARMY_SIZE-opp_army_size if who_has_largest_army == 0 else 0 if who_has_largest_army == -1 else army_diff+1 # 1 accounts for needing to beat the record rather than just match to claim it

    # Resources
    self_pip_production = get_player_production(game, SELF_COLOR, consider_robber=False)
    self_total_pips = sum(self_pip_production)
    opp_pip_production = get_player_production(game, OPP_COLOR, consider_robber=False)
    opp_total_pips = sum(opp_pip_production)
    pip_production_diff = [self_pip_production[i] - opp_pip_production[i] for i in range(len(self_pip_production))]

    # Robber
    self_robbed_pip_production = get_player_production(game, SELF_COLOR, consider_robber=True)
    opp_robbed_pip_production = get_player_production(game, OPP_COLOR, consider_robber=True)
    self_harm_by_robber = sum([self_pip_production[i] - self_robbed_pip_production[i] for i in range(len(self_pip_production))])
    opp_harm_by_robber = sum([opp_pip_production[i] - opp_robbed_pip_production[i] for i in range(len(opp_pip_production))])
    
    # Trade values
    self_ports = game.state.board.get_player_port_resources(SELF_COLOR)
    self_has_3_to_1_port = None in self_ports
    self_trade_exchange_rates = [1/2 if resource in self_ports else 1/3 if self_has_3_to_1_port else 1/4 for resource in RESOURCES]
    opp_ports = game.state.board.get_player_port_resources(OPP_COLOR)
    opp_has_3_to_1_port = None in opp_ports
    opp_trade_exchange_rates = [1/2 if resource in opp_ports else 1/3 if opp_has_3_to_1_port else 1/4 for resource in RESOURCES]

    # Diversity
    self_pip_diversity = compute_income_diversity(self_pip_production)
    opp_pip_diversity = compute_income_diversity(opp_pip_production)

    self_roll_diversity = compute_roll_diversity(game, SELF_COLOR)
    opp_roll_diversity = compute_roll_diversity(game, OPP_COLOR)

    return [
        # Victory Points
        self_VP_to_win, opp_VP_to_win, VP_diff,

        # Roads
        self_dist_to_longest_road, opp_dist_to_longest_road,
        road_length_diff, who_has_longest_road,

        # Army
        self_dist_to_largest_army, opp_dist_to_largest_army,
        army_diff, who_has_largest_army,

        # Production Pips
        *self_pip_production, self_total_pips,
        *opp_pip_production, opp_total_pips,
        *pip_production_diff,

        # Robber
        self_harm_by_robber, opp_harm_by_robber,

        # Trade
        *self_trade_exchange_rates,
        *opp_trade_exchange_rates,

        # Diversity
        self_pip_diversity, opp_pip_diversity,
        self_roll_diversity, opp_roll_diversity,
    ]
    
len(get_strategic_features(game))

44

#### Game State and Affordability

In [229]:
from catanatron.models.decks import starting_devcard_bank, ROAD_COST_FREQDECK, SETTLEMENT_COST_FREQDECK, CITY_COST_FREQDECK, DEVELOPMENT_CARD_COST_FREQDECK

In [218]:
from collections import Counter

In [242]:
def get_dev_card_features(game):
    
    # total_starting devcards
    dev_card_deck = starting_devcard_bank()
    dev_card_counts = Counter(dev_card_deck)

    # Total remaining in deck
    num_dev_cards_remaining = len(game.state.development_listdeck)
    
    player_state = game.state.player_state
    count_dev_cards_played = {}
    for dev_card in DEVELOPMENT_CARDS:
        if dev_card == VICTORY_POINT:
            continue
        
        played = 0
        for i in range(len(game.state.colors)): 
            played += player_state[f"P{i}_PLAYED_{dev_card}"]
        
        count_dev_cards_played[dev_card] = played

    pct_dev_remaining = num_dev_cards_remaining / len(dev_card_deck)

    pct_played = [count_dev_cards_played[dev_card]/dev_card_counts[dev_card] for dev_card in DEVELOPMENT_CARDS if dev_card != VICTORY_POINT] #
    
    return [pct_dev_remaining, *pct_played]

get_dev_card_features(game)

[0.16, 0.8571428571428571, 0.5, 0.5, 1.0]

In [239]:
def item_affordability(resources: list, costs: list)-> tuple[bool, int]:
    can_afford = True
    total_cost = sum(costs)
    needed_resources = 0
    for amt, cost in zip(resources, costs):
        if amt < cost:
            can_afford = False
        needed_resources += max(0, cost-amt)
    
    return can_afford, (total_cost - needed_resources)/total_cost

In [ ]:
def get_game_features(game):
    player_states = game.state.player_state
    color_to_index = game.state.color_to_index
    
    # turn number
    turn_num = game.state.num_turns

    # dev card stuff
    dev_card_features = get_dev_card_features(game)

    # affordability 
    self_index = color_to_index[SELF_COLOR]
    opp_index = color_to_index[OPP_COLOR]

    self_resource_counts = [player_states[f"P{self_index}_{resource}_IN_HAND"] for resource in RESOURCES]

    self_can_place_settlement = len(game.state.board.buildable_node_ids(SELF_COLOR)) > 0
    self_can_place_road = len(game.state.board.buildable_edges(SELF_COLOR)) > 0
    self_can_place_city = len(game.state.buildings_by_color[SELF_COLOR][SETTLEMENT]) > 0

    self_can_afford_road, self_pct_road_afforded = item_affordability(self_resource_counts, ROAD_COST_FREQDECK)
    self_can_afford_settlement, self_pct_settlement_afforded = item_affordability(self_resource_counts, SETTLEMENT_COST_FREQDECK)
    self_can_afford_city, self_pct_city_afforded = item_affordability(self_resource_counts, CITY_COST_FREQDECK)
    self_can_afford_dev, self_pct_dev_afforded = item_affordability(self_resource_counts, DEVELOPMENT_CARD_COST_FREQDECK)
    

    opp_resource_counts = [player_states[f"P{opp_index}_{resource}_IN_HAND"] for resource in RESOURCES]

    opp_can_place_settlement = len(game.state.board.buildable_node_ids(OPP_COLOR)) > 0
    opp_can_place_road = len(game.state.board.buildable_edges(OPP_COLOR)) > 0
    opp_can_place_city = len(game.state.buildings_by_color[OPP_COLOR][SETTLEMENT]) > 0

    opp_can_afford_road, opp_pct_road_afforded = item_affordability(opp_resource_counts, ROAD_COST_FREQDECK)
    opp_can_afford_settlement, opp_pct_settlement_afforded = item_affordability(opp_resource_counts, SETTLEMENT_COST_FREQDECK)
    opp_can_afford_city, opp_pct_city_afforded = item_affordability(opp_resource_counts, CITY_COST_FREQDECK)
    opp_can_afford_dev, opp_pct_dev_afforded = item_affordability(opp_resource_counts, DEVELOPMENT_CARD_COST_FREQDECK)

    return [
        # Turn number
        turn_num / 100,

        # Dev card features
        *dev_card_features,

        # Self affordability - can place
        float(self_can_place_settlement),
        float(self_can_place_road),
        float(self_can_place_city),

        # Self affordability - can afford
        float(self_can_afford_road),
        float(self_can_afford_settlement),
        float(self_can_afford_city),
        float(self_can_afford_dev),

        # Self affordability - percent afforded
        self_pct_road_afforded,
        self_pct_settlement_afforded,
        self_pct_city_afforded,
        self_pct_dev_afforded,

        # Opp affordability - can place
        float(opp_can_place_settlement),
        float(opp_can_place_road),
        float(opp_can_place_city),

        # Opp affordability - can afford
        float(opp_can_afford_road),
        float(opp_can_afford_settlement),
        float(opp_can_afford_city),
        float(opp_can_afford_dev),

        # Opp affordability - percent afforded
        opp_pct_road_afforded,
        opp_pct_settlement_afforded,
        opp_pct_city_afforded,
        opp_pct_dev_afforded,
    ]
    